# 1. Setup

In [ ]:
import os, gc, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import torch
from tqdm.auto import tqdm

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (12, 5)

In [ ]:
SEED = 42
N_CLASSES = 19
ID_COL, LABEL_COL, TARGET_COL = "id", "label", "target_feature"

DATA_DIR = Path("/kaggle/input/competitions/3rd-wear-dataset-challenge-hasca-2026")
TRAIN_DIR, TEST_DIR = DATA_DIR / "train", DATA_DIR / "test"
TEST_INERTIAL_PATH = TEST_DIR / "test_inertial_data.npy"
TEST_VIDEO_PATH = TEST_DIR / "test_videomae_data.npy"
TEST_META_PATH = TEST_DIR / "test_meta_data.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"
WORK_DIR = Path("/kaggle/working")

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.benchmark = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPUS = torch.cuda.device_count()
USE_AMP = DEVICE.type == "cuda"
NUM_WORKERS = min(8, os.cpu_count() or 2)

print(f"PyTorch {torch.__version__} | Device: {DEVICE} | GPUs: {N_GPUS} | Workers: {NUM_WORKERS}")
for i in range(N_GPUS):
    p = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {p.name}, {p.total_memory / 1024**3:.1f} GiB, compute capability {p.major}.{p.minor}")

# 2. Data analysis

## 2.1 Dataset structure

In [ ]:
train_csv_paths = sorted(TRAIN_DIR.rglob("*.csv"))
train_npy_paths = sorted(TRAIN_DIR.rglob("*.npy"))

video_shapes, video_dtypes = [], []
for path in train_npy_paths:
    array = np.load(path, mmap_mode="r")
    video_shapes.append(str(array.shape)); video_dtypes.append(str(array.dtype))

recording_files = pd.DataFrame({
    "inertial_file": [str(p.relative_to(TRAIN_DIR)) for p in train_csv_paths],
    "inertial_mb": [p.stat().st_size / 1024**2 for p in train_csv_paths],
    "video_file": [str(p.relative_to(TRAIN_DIR)) for p in train_npy_paths],
    "video_mb": [p.stat().st_size / 1024**2 for p in train_npy_paths],
    "video_shape": video_shapes,
    "video_dtype": video_dtypes
})
display(recording_files.round({"inertial_mb": 1, "video_mb": 1}))

In [ ]:
example_path = train_csv_paths[0]
example_train = pd.read_csv(example_path, nrows=8)

print(f"Example recording: {example_path.name}")
display(example_train)
display(example_train.dtypes.rename("dtype").to_frame())

In [ ]:
test_inertial = np.load(TEST_INERTIAL_PATH, mmap_mode="r")
test_video = np.load(TEST_VIDEO_PATH, mmap_mode="r")
test_meta = pd.read_csv(TEST_META_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Inertial: {test_inertial.shape}, {test_inertial.dtype}")
print(f"Video:    {test_video.shape}, {test_video.dtype}")
display(test_meta.head())
display(pd.crosstab(test_meta["sbj_id"], test_meta["sensor_location"], margins=True))

## 2.2 Recording duration and label distribution

In [ ]:
video_by_stem = {p.stem: p for p in train_npy_paths}
recording_rows, label_rows = [], []

for csv_path in tqdm(train_csv_paths, desc="Reading recording labels"):
    labels = pd.read_csv(csv_path, usecols=["sbj_id", "label"], dtype={"sbj_id": "int16", "label": "string"}, memory_map=True)
    video = np.load(video_by_stem[csv_path.stem], mmap_mode="r")
    counts = labels["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="samples")
    counts["recording"] = csv_path.stem; label_rows.append(counts)
    recording_rows.append({
        "recording": csv_path.stem, "sbj_id": int(labels["sbj_id"].iat[0]), "inertial_rows": len(labels), "video_frames": video.shape[0],
        "inertial_min": len(labels) / 50 / 60, "video_min": video.shape[0] / 30 / 60,
        "duration_gap_s": len(labels) / 50 - video.shape[0] / 30, "non_missing_labels_pct": labels["label"].notna().mean() * 100
    })

recording_summary = pd.DataFrame(recording_rows)
raw_label_counts = pd.concat(label_rows, ignore_index=True)
display(recording_summary.round({"inertial_min": 2, "video_min": 2, "duration_gap_s": 3, "non_missing_labels_pct": 1}))

In [ ]:
label_summary = raw_label_counts.groupby("label", dropna=False, as_index=False)["samples"].sum()
label_summary["activity"] = label_summary["label"].astype("string").fillna("null")
label_summary["hours"] = label_summary["samples"] / 50 / 3600
label_summary["share_pct"] = label_summary["samples"] / label_summary["samples"].sum() * 100
label_summary = label_summary.sort_values("samples", ascending=False).reset_index(drop=True)

display(label_summary[["activity", "samples", "hours", "share_pct"]].round({"hours": 2, "share_pct": 2}))

plt.figure(figsize=(10, 7))
ax = sns.barplot(data=label_summary.sort_values("hours"), x="hours", y="activity", color=sns.color_palette()[0])
ax.set(xlabel="Recording hours", ylabel="", title="Training label distribution")
plt.show()

In [ ]:
CLASS_NAMES = ["null", "jogging", "jogging (rotating arms)", "jogging (skipping)", "jogging (sidesteps)", "jogging (butt-kicks)",
               "stretching (triceps)", "stretching (lunging)", "stretching (shoulders)", "stretching (hamstrings)", "stretching (lumbar rotation)",
               "push-ups", "push-ups (complex)", "sit-ups", "sit-ups (complex)", "burpees", "lunges", "lunges (complex)", "bench-dips"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}

subject_activity = raw_label_counts.copy()
subject_activity["activity"] = subject_activity["label"].astype("string").fillna("null")
subject_activity["sbj_id"] = subject_activity["recording"].map(recording_summary.set_index("recording")["sbj_id"])
subject_activity = subject_activity.groupby(["sbj_id", "activity"], as_index=False)["samples"].sum()
subject_activity["minutes"] = subject_activity["samples"] / 50 / 60

subject_summary = recording_summary.groupby("sbj_id", as_index=False).agg(recordings=("recording", "count"), minutes=("inertial_min", "sum"))
null_share = subject_activity[subject_activity["activity"] == "null"].set_index("sbj_id")["samples"] / subject_activity.groupby("sbj_id")["samples"].sum()
coverage = subject_activity[subject_activity["activity"] != "null"].groupby("sbj_id")["activity"].nunique()
subject_summary["null_pct"] = subject_summary["sbj_id"].map(null_share).mul(100)
subject_summary["activity_classes"] = subject_summary["sbj_id"].map(coverage)
display(subject_summary.round({"minutes": 1, "null_pct": 1}))

activity_by_subject = subject_activity.pivot(index="sbj_id", columns="activity", values="minutes").reindex(index=range(22), columns=CLASS_NAMES).fillna(0).astype(float)

plt.figure(figsize=(16, 8))
sns.heatmap(activity_by_subject.T, cmap="Blues")
plt.xlabel("Training participant"); plt.ylabel(""); plt.title("Activity minutes by participant")
plt.show()

In [ ]:
segment_parts = []

for csv_path in tqdm(train_csv_paths, desc="Finding activity segments"):
    labels = pd.read_csv(csv_path, usecols=["label"], dtype={"label": "string"}, memory_map=True)["label"].fillna("null").to_numpy(dtype=str)
    starts = np.r_[0, np.flatnonzero(labels[1:] != labels[:-1]) + 1]
    lengths = np.diff(np.r_[starts, len(labels)])
    segment_parts.append(pd.DataFrame({"recording": csv_path.stem, "activity": labels[starts], "seconds": lengths / 50}))

segments = pd.concat(segment_parts, ignore_index=True)
segment_summary = segments.groupby("activity", as_index=False).agg(
    segments=("seconds", "size"), median_s=("seconds", "median"), p10_s=("seconds", lambda x: x.quantile(.1)),
    p90_s=("seconds", lambda x: x.quantile(.9)), segments_ge_1s_pct=("seconds", lambda x: (x >= 1).mean() * 100)
)
segment_summary["activity"] = pd.Categorical(segment_summary["activity"], categories=CLASS_NAMES, ordered=True)
display(segment_summary.sort_values("activity").round(2))

## 2.3 Inertial signals

In [ ]:
SENSOR_LOCATIONS = ["right_arm", "right_leg", "left_leg", "left_arm"]
SENSOR_COLUMNS = {sensor: [f"{sensor}_acc_{axis}" for axis in "xyz"] for sensor in SENSOR_LOCATIONS}
EXAMPLE_ACTIVITIES = ["null", "jogging", "stretching (triceps)", "push-ups", "lunges"]

segments["samples"] = (segments["seconds"] * 50).round().astype(int)
segments["end"] = segments.groupby("recording", sort=False)["samples"].cumsum()
segments["start"] = segments["end"] - segments["samples"]

selected_segments = segments[segments["activity"].isin(EXAMPLE_ACTIVITIES)].sort_values("samples").groupby("activity", sort=False).tail(1)
selected_segments = selected_segments.set_index("activity").loc[EXAMPLE_ACTIVITIES].reset_index()
display(selected_segments[["activity", "recording", "seconds"]].round({"seconds": 1}))

inertial_by_stem = {p.stem: p for p in train_csv_paths}
recording_cache, example_parts = {}, []

for row in selected_segments.itertuples():
    if row.recording not in recording_cache:
        recording_cache[row.recording] = pd.read_csv(inertial_by_stem[row.recording], usecols=sum(SENSOR_COLUMNS.values(), []), dtype=np.float32)
    start = int(row.start + (row.samples - 50) // 2)
    for sensor, columns in SENSOR_COLUMNS.items():
        values = recording_cache[row.recording].iloc[start:start + 50][columns].to_numpy()
        example_parts.append(pd.DataFrame({"time_s": np.arange(50) / 50, "magnitude": np.linalg.norm(values, axis=1), "activity": row.activity, "sensor": sensor.replace("_", " ")}))

inertial_examples = pd.concat(example_parts, ignore_index=True)
del recording_cache
gc.collect()

In [ ]:
g = sns.relplot(
    data=inertial_examples, x="time_s", y="magnitude", row="activity", col="sensor", kind="line",
    row_order=EXAMPLE_ACTIVITIES, col_order=[s.replace("_", " ") for s in SENSOR_LOCATIONS],
    height=1.7, aspect=1.35, facet_kws={"sharey": True}, errorbar=None
)
g.set_axis_labels("Time (s)", "Acceleration magnitude")
g.set_titles("{row_name} | {col_name}")
g.figure.suptitle("Representative one-second inertial windows", y=1.01)
plt.show()

## 2.4 Video features

In [ ]:
WINDOW_SAMPLES, EDA_STRIDE, EDA_WINDOWS_PER_CLASS = 50, 250, 80
recording_to_subject = recording_summary.set_index("recording")["sbj_id"].to_dict()
window_rows = []

for row in segments.itertuples(index=False):
    first_start = ((row.start + 49) // 50) * 50
    starts = np.arange(first_start, row.end - WINDOW_SAMPLES + 1, EDA_STRIDE, dtype=np.int64)
    window_rows.extend({"recording": row.recording, "sbj_id": recording_to_subject[row.recording], "activity": row.activity, "start": int(start)} for start in starts)

eda_candidates = pd.DataFrame(window_rows)
eda_windows = pd.concat([group.sample(min(EDA_WINDOWS_PER_CLASS, len(group)), random_state=SEED) for _, group in eda_candidates.groupby("activity")], ignore_index=True)
eda_windows["target"] = eda_windows["activity"].map(LABEL_TO_ID)

eda_window_summary = eda_windows.groupby("activity").agg(windows=("start", "size"), participants=("sbj_id", "nunique")).reindex(CLASS_NAMES)
display(eda_window_summary)

In [ ]:
video_cache, train_video_pooled, video_stat_rows = {}, [], []

for row in tqdm(eda_windows.itertuples(index=False), total=len(eda_windows), desc="Sampling train video"):
    if row.recording not in video_cache:
        video_cache[row.recording] = np.load(video_by_stem[row.recording], mmap_mode="r")
    video_start = row.start * 3 // 5
    clip = np.asarray(video_cache[row.recording][video_start + 8:video_start + 23], dtype=np.float32)
    frame_norms = np.linalg.norm(clip, axis=1); normalized = clip / (frame_norms[:, None] + 1e-8)
    pooled = clip.mean(axis=0); train_video_pooled.append(pooled)
    video_stat_rows.append({"split": "train", "pooled_norm": np.linalg.norm(pooled), "temporal_std": clip.std(axis=0).mean(), "adjacent_cosine": (normalized[:-1] * normalized[1:]).sum(axis=1).mean()})

rng = np.random.default_rng(SEED)
test_indices = rng.choice(len(test_video), size=min(len(eda_windows), len(test_video)), replace=False)

for idx in tqdm(test_indices, desc="Sampling test video"):
    clip = np.asarray(test_video[idx].T, dtype=np.float32)
    frame_norms = np.linalg.norm(clip, axis=1); normalized = clip / (frame_norms[:, None] + 1e-8)
    pooled = clip.mean(axis=0)
    video_stat_rows.append({"split": "test", "pooled_norm": np.linalg.norm(pooled), "temporal_std": clip.std(axis=0).mean(), "adjacent_cosine": (normalized[:-1] * normalized[1:]).sum(axis=1).mean()})

train_video_pooled = np.stack(train_video_pooled)
video_stats = pd.DataFrame(video_stat_rows)
display(video_stats.groupby("split")[["pooled_norm", "temporal_std", "adjacent_cosine"]].agg(["mean", "std"]).round(4))
del video_cache

In [ ]:
video_stats_long = video_stats.melt(id_vars="split", var_name="metric", value_name="value")
g = sns.displot(data=video_stats_long, x="value", hue="split", col="metric", kind="kde", common_norm=False,
                facet_kws={"sharex": False, "sharey": False}, height=3.2, aspect=1.15)
g.set_titles("{col_name}")
g.figure.suptitle("Train and test VideoMAE feature distributions", y=1.05)
plt.show()

## 2.5 Train and test inertial distributions

In [ ]:
domain_windows = eda_candidates.sample(min(3000, len(eda_candidates)), random_state=SEED)
train_stat_parts = []

for recording, group in tqdm(domain_windows.groupby("recording"), total=domain_windows["recording"].nunique(), desc="Sampling train inertial"):
    data = pd.read_csv(inertial_by_stem[recording], usecols=sum(SENSOR_COLUMNS.values(), []), dtype=np.float32, memory_map=True)
    starts = group["start"].to_numpy()
    for sensor, columns in SENSOR_COLUMNS.items():
        windows = np.stack([data.iloc[start:start + 50][columns].to_numpy() for start in starts])
        magnitude = np.linalg.norm(windows, axis=2)
        train_stat_parts.append(pd.DataFrame({
            "split": "train", "sensor_location": sensor, "mean_magnitude": magnitude.mean(axis=1),
            "dynamic_rms": np.sqrt(np.mean((windows - windows.mean(axis=1, keepdims=True)) ** 2, axis=(1, 2))),
            "jerk_rms": np.sqrt(np.mean(np.diff(windows, axis=1) ** 2, axis=(1, 2)))
        }))

train_inertial_stats = pd.concat(train_stat_parts, ignore_index=True)
test_array = np.asarray(test_inertial, dtype=np.float32)
test_magnitude = np.linalg.norm(test_array, axis=2)
test_inertial_stats = pd.DataFrame({
    "split": "test", "sensor_location": test_meta["sensor_location"], "mean_magnitude": test_magnitude.mean(axis=1),
    "dynamic_rms": np.sqrt(np.mean((test_array - test_array.mean(axis=1, keepdims=True)) ** 2, axis=(1, 2))),
    "jerk_rms": np.sqrt(np.mean(np.diff(test_array, axis=1) ** 2, axis=(1, 2)))
})

inertial_stats = pd.concat([train_inertial_stats, test_inertial_stats], ignore_index=True)
display(inertial_stats.groupby(["split", "sensor_location"])[["mean_magnitude", "dynamic_rms", "jerk_rms"]].mean().round(4))

In [ ]:
inertial_stats_long = inertial_stats.melt(id_vars=["split", "sensor_location"], var_name="metric", value_name="value")
inertial_stats_long["sensor_location"] = inertial_stats_long["sensor_location"].str.replace("_", " ")

g = sns.catplot(
    data=inertial_stats_long, x="sensor_location", y="value", hue="split", col="metric", kind="box",
    order=[s.replace("_", " ") for s in SENSOR_LOCATIONS], hue_order=["train", "test"],
    showfliers=False, sharey=False, height=3.5, aspect=1.15
)
g.set_axis_labels("", "Value"); g.set_titles("{col_name}")
g.figure.suptitle("Train and test inertial distributions", y=1.04)
plt.show()

- The training set contains 24 synchronized inertial-video recordings from 22 participants. Test data contains four entirely unseen participants, making participant-independent validation essential.
- Inertial and video streams are aligned exactly at 50 Hz and 30 FPS. A one-second training window corresponds to 50 inertial samples and 30 video frames. VideoMAE indices 8-22 provide the 15 valid features available in test data.
- Missing training labels represent the `null` class. It occupies 39.7% of the timeline, while the remaining 18 activities are relatively balanced.
- Activity segments are much longer than one second, allowing pure-label windows. Overlapping windows remain highly correlated and must never be randomly divided between training and validation.
- Jogging produces strong periodic inertial patterns, while slow and posture-based activities can appear nearly stationary within a one-second window. Raw axes, sensor location and visual features should therefore complement acceleration magnitude.
- Train and test feature distributions are similar for both modalities. Per-sensor inertial normalization and normalization inside the video branch should handle the remaining differences.
- Consecutive VideoMAE features are highly correlated, suggesting that statistical pooling or lightweight temporal attention may be more appropriate than a large video Transformer.

# 4. Training pipeline

## 4.1 Configuration

In [ ]:
MODE = "full"  # smoke / full

CONFIG = {
    "smoke": {
        "windows_per_subject_class": 2,
        "pooled_epochs": 1,
        "temporal_epochs": 1,
    },
    "full": {
        "windows_per_subject_class": 150,
        "pooled_epochs": 3,
        "temporal_epochs": 4,
    },
}[MODE]

WINDOW_STRIDE = 25

WINDOWS_PER_SUBJECT_CLASS = CONFIG["windows_per_subject_class"]
POOLED_EPOCHS, TEMPORAL_EPOCHS = CONFIG["pooled_epochs"], CONFIG["temporal_epochs"]
TRAIN_BATCH_SIZE, TEST_BATCH_SIZE = 256, 512
TEMPORAL_WEIGHT = 0.60
NULL_BIAS = 0.75
N_CLASSES, ID_COL, TARGET_COL = 19, "id", "target_feature"

CLASS_NAMES = ["null", "jogging", "jogging (rotating arms)", "jogging (skipping)", "jogging (sidesteps)", "jogging (butt-kicks)",
               "stretching (triceps)", "stretching (lunging)", "stretching (shoulders)", "stretching (hamstrings)", "stretching (lumbar rotation)",
               "push-ups", "push-ups (complex)", "sit-ups", "sit-ups (complex)", "burpees", "lunges", "lunges (complex)", "bench-dips"]
LABEL_TO_ID = {name: i for i, name in enumerate(CLASS_NAMES)}
SENSOR_LOCATIONS = ["right_arm", "right_leg", "left_leg", "left_arm"]
SENSOR_TO_ID = {name: i for i, name in enumerate(SENSOR_LOCATIONS)}
SENSOR_COLUMNS = {sensor: [f"{sensor}_acc_{axis}" for axis in "xyz"] for sensor in SENSOR_LOCATIONS}

print(f"Mode: {MODE} | Pooled epochs: {POOLED_EPOCHS} | Temporal epochs: {TEMPORAL_EPOCHS}")

In [ ]:
train_csv_paths = sorted(TRAIN_DIR.rglob("*.csv"))
train_npy_paths = sorted(TRAIN_DIR.rglob("*.npy"))
video_by_stem = {p.stem: p for p in train_npy_paths}
CACHE_DIR = WORK_DIR / "inertial_cache"
CACHE_DIR.mkdir(exist_ok=True)
window_rows = []

for csv_path in tqdm(train_csv_paths, desc="Preparing training data"):
    recording, cache_path = csv_path.stem, CACHE_DIR / f"{csv_path.stem}.npy"
    if cache_path.exists():
        labels_frame = pd.read_csv(csv_path, usecols=["sbj_id", "label"], dtype={"sbj_id": "int16", "label": "string"}, memory_map=True)
    else:
        sensor_columns = sum(SENSOR_COLUMNS.values(), [])
        dtype_map = {"sbj_id": "int16", "label": "string", **{column: "float32" for column in sensor_columns}}
        data = pd.read_csv(csv_path, usecols=["sbj_id", "label"] + sensor_columns, dtype=dtype_map, memory_map=True)
        inertial = np.stack([data[columns].to_numpy(copy=False) for columns in SENSOR_COLUMNS.values()], axis=1)
        np.save(cache_path, inertial)
        labels_frame = data[["sbj_id", "label"]].copy()
        del data, inertial

    sbj_id = int(labels_frame["sbj_id"].iat[0])
    labels = labels_frame["label"].fillna("null").to_numpy(dtype=str)
    boundaries = np.flatnonzero(labels[1:] != labels[:-1]) + 1
    starts, ends = np.r_[0, boundaries], np.r_[boundaries, len(labels)]

    for start, end in zip(starts, ends):
        first_start = (
            (start + WINDOW_STRIDE - 1) // WINDOW_STRIDE
        ) * WINDOW_STRIDE  
        window_starts = np.arange(
            first_start, end - 50 + 1, WINDOW_STRIDE,
            dtype=np.int64,
        )
        target = LABEL_TO_ID[labels[start]]
        window_rows.extend({
            "recording": recording, "sbj_id": sbj_id, "target": target,
            "inertial_start": int(window_start), "video_start": int(window_start * 3 // 5)
        } for window_start in window_starts)

window_index = pd.DataFrame(window_rows)
cache_paths = {p.stem: p for p in CACHE_DIR.glob("*.npy")}

In [ ]:
train_index = pd.concat([
    group.sample(min(WINDOWS_PER_SUBJECT_CLASS, len(group)), random_state=SEED)
    for _, group in window_index.groupby(["sbj_id", "target"])
], ignore_index=True)

train_index = train_index.sample(frac=1, random_state=SEED).reset_index(drop=True)
print(f"Temporal windows: {len(train_index):,} | Sensor samples: {len(train_index) * 4:,}")

## 4.2 Dataset

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F

class WearTrainDataset(Dataset):
    def __init__(self, index):
        self.recordings = index["recording"].to_numpy()
        self.inertial_starts = index["inertial_start"].to_numpy()
        self.video_starts = index["video_start"].to_numpy()
        self.targets = index["target"].to_numpy()
        self.inertial_cache, self.video_cache = {}, {}

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        recording = self.recordings[idx]
        if recording not in self.inertial_cache:
            self.inertial_cache[recording] = np.load(cache_paths[recording], mmap_mode="r")
            self.video_cache[recording] = np.load(video_by_stem[recording], mmap_mode="r")

        inertial_start, video_start = int(self.inertial_starts[idx]), int(self.video_starts[idx])
        inertial = self.inertial_cache[recording][inertial_start:inertial_start + 50].transpose(1, 2, 0).copy()
        valid_sensors = np.isfinite(inertial).all(axis=(1, 2))
        np.nan_to_num(inertial, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        video = self.video_cache[recording][video_start + 8:video_start + 23].copy()
        return inertial, video, int(self.targets[idx]), valid_sensors

train_dataset = WearTrainDataset(train_index)
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

## 4.3 Ensemble models

In [ ]:
class PooledFusionModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.inertial_encoder = nn.Sequential(nn.Linear(16, 64), nn.LayerNorm(64), nn.GELU(), nn.Dropout(.15))
        self.video_encoder = nn.Sequential(nn.LayerNorm(1536), nn.Linear(1536, 192), nn.GELU(), nn.Dropout(.2))
        self.sensor_embedding = nn.Embedding(4, 8)
        self.classifier = nn.Sequential(nn.Linear(264, 128), nn.GELU(), nn.Dropout(.2), nn.Linear(128, N_CLASSES))

    def forward(self, inertial, video, sensor_ids=None):
        batch_size, sensors = inertial.shape[:2]
        magnitude = torch.linalg.vector_norm(inertial, dim=2)
        inertial_features = torch.cat([
            inertial.mean(-1), inertial.std(-1, unbiased=False), inertial.amin(-1), inertial.amax(-1),
            magnitude.mean(-1, keepdim=True), magnitude.std(-1, unbiased=False, keepdim=True),
            magnitude.amin(-1, keepdim=True), magnitude.amax(-1, keepdim=True)
        ], dim=-1)
        inertial_features = self.inertial_encoder(inertial_features)
        video_features = self.video_encoder(torch.cat([video.mean(1), video.std(1, unbiased=False)], dim=-1))
        video_features = video_features[:, None].expand(-1, sensors, -1)

        if sensor_ids is None:
            sensor_ids = torch.arange(sensors, device=inertial.device)[None].expand(batch_size, -1)

        fused = torch.cat([inertial_features, video_features, self.sensor_embedding(sensor_ids)], dim=-1)
        return self.classifier(fused).reshape(-1, N_CLASSES)

In [ ]:
class InceptionBlock(nn.Module):
    def __init__(self, in_channels, branch_channels=32):
        super().__init__()
        out_channels = branch_channels * 4
        self.bottleneck = nn.Conv1d(in_channels, branch_channels, 1, bias=False)
        self.branches = nn.ModuleList([nn.Conv1d(branch_channels, branch_channels, kernel, padding=kernel // 2, bias=False) for kernel in (5, 11, 21)])
        self.pool_branch = nn.Sequential(nn.MaxPool1d(3, stride=1, padding=1), nn.Conv1d(in_channels, branch_channels, 1, bias=False))
        self.skip = nn.Conv1d(in_channels, out_channels, 1, bias=False) if in_channels != out_channels else nn.Identity()
        self.norm = nn.GroupNorm(8, out_channels)
        self.dropout = nn.Dropout(.1)

    def forward(self, x):
        reduced = self.bottleneck(x)
        merged = torch.cat([branch(reduced) for branch in self.branches] + [self.pool_branch(x)], dim=1)
        return self.dropout(F.gelu(self.norm(merged + self.skip(x))))

class InertialEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.blocks = nn.Sequential(InceptionBlock(4), InceptionBlock(128))
        self.head = nn.Sequential(nn.Linear(256, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(.2))

    def forward(self, x):
        magnitude = torch.linalg.vector_norm(x, dim=1, keepdim=True)
        x = self.blocks(torch.cat([x, magnitude], dim=1))
        return self.head(torch.cat([x.mean(-1), x.amax(-1)], dim=1))

class VideoEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.projection = nn.Sequential(nn.LayerNorm(768), nn.Linear(768, 192), nn.GELU())
        self.position = nn.Parameter(torch.zeros(1, 15, 192))
        layer = nn.TransformerEncoderLayer(192, 4, 384, dropout=.15, activation="gelu", batch_first=True, norm_first=True)
        self.temporal = nn.TransformerEncoder(layer, 1, enable_nested_tensor=False)
        self.attention = nn.Sequential(nn.Linear(192, 64), nn.Tanh(), nn.Linear(64, 1))
        self.head = nn.Sequential(nn.Linear(384, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(.2))
        nn.init.trunc_normal_(self.position, std=.02)

    def forward(self, x):
        x = self.temporal(self.projection(x) + self.position)
        weights = torch.softmax(self.attention(x), dim=1)
        return self.head(torch.cat([(x * weights).sum(dim=1), x.mean(dim=1)], dim=1))

class TemporalFusionModel(nn.Module):
    def __init__(self, modality_dropout=.1):
        super().__init__()
        self.inertial_encoder, self.video_encoder = InertialEncoder(), VideoEncoder()
        self.sensor_embedding = nn.Embedding(4, 16)
        self.gate = nn.Linear(400, 192)
        self.classifier = nn.Sequential(nn.Linear(400, 192), nn.LayerNorm(192), nn.GELU(), nn.Dropout(.3), nn.Linear(192, N_CLASSES))
        self.modality_dropout = modality_dropout

    def forward(self, inertial, video, sensor_ids=None):
        batch_size, sensors = inertial.shape[:2]
        if self.training:
            scale = torch.empty(batch_size, sensors, 1, 1, device=inertial.device).uniform_(.9, 1.1)
            inertial = inertial * scale + torch.randn_like(inertial) * .01

        inertial_features = self.inertial_encoder(inertial.reshape(-1, 3, 50)).reshape(batch_size, sensors, -1)
        video_features = self.video_encoder(video)

        if self.training and self.modality_dropout:
            inertial_keep = (torch.rand(batch_size, sensors, 1, device=inertial.device) > self.modality_dropout) / (1 - self.modality_dropout)
            video_keep = (torch.rand(batch_size, 1, device=video.device) > self.modality_dropout) / (1 - self.modality_dropout)
            inertial_features = inertial_features * inertial_keep
            video_features = video_features * video_keep

        video_features = video_features[:, None].expand(-1, sensors, -1)
        if sensor_ids is None:
            sensor_ids = torch.arange(sensors, device=inertial.device)[None].expand(batch_size, -1)

        sensor_features = self.sensor_embedding(sensor_ids)
        gate = torch.sigmoid(self.gate(torch.cat([inertial_features, video_features, sensor_features], dim=-1)))
        fused = gate * inertial_features + (1 - gate) * video_features
        features = torch.cat([fused, inertial_features * video_features, sensor_features], dim=-1)
        return self.classifier(features).reshape(-1, N_CLASSES)

## 4.4 Training

In [ ]:
def fit_model(model, loader, epochs, lr, weight_decay, name, cosine_tmax=None):
    criterion = nn.CrossEntropyLoss(label_smoothing=.05)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cosine_tmax, eta_min=lr * .1) if cosine_tmax else None
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
    history = []

    for epoch in range(1, epochs + 1):
        model.train(); total_loss, total_items = 0, 0
        for inertial, video, target, valid in tqdm(loader, desc=f"{name} {epoch}/{epochs}", leave=False):
            inertial = inertial.to(DEVICE, non_blocking=True); video = video.to(DEVICE, non_blocking=True)
            target = target.to(DEVICE, non_blocking=True).repeat_interleave(inertial.shape[1])
            valid = valid.to(DEVICE, non_blocking=True).reshape(-1)
            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
                logits = model(inertial, video)
                loss = criterion(logits[valid], target[valid])

            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            items = valid.sum().item(); total_loss += loss.item() * items; total_items += items

        if scheduler:
            scheduler.step()

        epoch_loss = total_loss / total_items
        history.append({"epoch": epoch, "train_loss": epoch_loss})
        print(f"{name} epoch {epoch}/{epochs}: loss {epoch_loss:.4f}")

    return pd.DataFrame(history)

In [ ]:
pooled_model = PooledFusionModel().to(DEVICE)
pooled_history = fit_model(pooled_model, train_loader, POOLED_EPOCHS, lr=2e-3, weight_decay=1e-4, name="Pooled model")
torch.save(pooled_model.state_dict(), WORK_DIR / f"pooled_model_{MODE}.pt")
display(pooled_history)

In [ ]:
temporal_model = TemporalFusionModel().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in temporal_model.parameters())

if N_GPUS > 1:
    temporal_model = nn.DataParallel(temporal_model)

print(f"Temporal model parameters: {parameter_count / 1e6:.2f}M | GPUs: {N_GPUS if isinstance(temporal_model, nn.DataParallel) else 1}")

temporal_history = fit_model(temporal_model, train_loader, TEMPORAL_EPOCHS, lr=8e-4, weight_decay=1e-3, name="Temporal model", cosine_tmax=10)
temporal_state = temporal_model.module.state_dict() if isinstance(temporal_model, nn.DataParallel) else temporal_model.state_dict()
torch.save(temporal_state, WORK_DIR / f"temporal_model_{MODE}.pt")
display(temporal_history)

# 5. Submission

In [ ]:
class WearTestDataset(Dataset):
    def __init__(self):
        self.meta = pd.read_csv(TEST_META_PATH)
        self.ids = self.meta[ID_COL].to_numpy()
        self.sensor_ids = self.meta["sensor_location"].map(SENSOR_TO_ID).to_numpy(dtype=np.int64)
        self.inertial, self.video = None, None

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        if self.inertial is None:
            self.inertial = np.load(TEST_INERTIAL_PATH, mmap_mode="r")
            self.video = np.load(TEST_VIDEO_PATH, mmap_mode="r")

        inertial = np.asarray(self.inertial[idx], dtype=np.float32).T[None].copy()
        np.nan_to_num(inertial, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
        video = np.asarray(self.video[idx], dtype=np.float32).T.copy()
        return int(self.ids[idx]), inertial, video, int(self.sensor_ids[idx])

test_dataset = WearTestDataset()
test_loader = DataLoader(test_dataset, batch_size=TEST_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)

In [ ]:
@torch.inference_mode()
def predict_test(pooled_model, temporal_model, loader):
    pooled_model.eval(); temporal_model.eval()
    ids, pooled_parts, temporal_parts = [], [], []

    for batch_ids, inertial, video, sensor_ids in tqdm(loader, desc="Predicting test data"):
        inertial = inertial.to(DEVICE, non_blocking=True); video = video.to(DEVICE, non_blocking=True)
        sensor_ids = sensor_ids.to(DEVICE, non_blocking=True).long().unsqueeze(1)

        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
            pooled_logits = pooled_model(inertial, video, sensor_ids)
            temporal_logits = temporal_model(inertial, video, sensor_ids)

        ids.append(batch_ids.numpy())
        pooled_parts.append(torch.softmax(pooled_logits.float(), dim=1).cpu().numpy())
        temporal_parts.append(torch.softmax(temporal_logits.float(), dim=1).cpu().numpy())

    return np.concatenate(ids), np.concatenate(pooled_parts), np.concatenate(temporal_parts)

test_ids, pooled_test_probs, temporal_test_probs = predict_test(pooled_model, temporal_model, test_loader)

In [ ]:
combined_probs = TEMPORAL_WEIGHT * temporal_test_probs + (1 - TEMPORAL_WEIGHT) * pooled_test_probs
combined_probs[:, 0] *= np.exp(NULL_BIAS)
test_predictions = combined_probs.argmax(axis=1).astype(int)

prediction_map = pd.Series(test_predictions, index=test_ids)
submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)
submission[TARGET_COL] = submission[ID_COL].map(prediction_map).astype(int)

SUBMISSION_PATH = WORK_DIR / "submission.csv"
submission.to_csv(SUBMISSION_PATH, index=False)

display(submission.head())
display(submission[TARGET_COL].value_counts().sort_index().rename("predictions").to_frame())
print(f"Submission saved to: {SUBMISSION_PATH}")